In [52]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotter_lib as pl
from scipy.stats import kendalltau
import os,json
from collections import defaultdict


ligas = ["England","France","Germany", "Italy", "Spain"]

ligaLabels = { liga:liga[:2].upper()  for liga in ligas}

labels=["EN","FR","GE","IT","SP"]

ligaColor = {"England":"#9e0142", "France":"#f46d43", "Germany":"#66c2a5", "Spain":"#5e4fa2", "Italy":"#3288bd"}


In [2]:
arx_matches = "/home/chacoma/Lineas/Futbol/Ranking/ranking-futbol/data/matches.csv"

df = pd.read_csv(arx_matches)

df.head()

,match,home,away,winner,score_home,score_away,label,liga
0,2500089,1646,1659,1659,1,2,"Burnley - AFC Bournemouth, 1 - 2",England
1,2500090,1628,1627,1628,2,0,"Crystal Palace - West Bromwich Albion, 2 - 0",England
2,2500091,1673,1609,1609,0,1,"Huddersfield Town - Arsenal, 0 - 1",England
3,2500092,1612,1651,1612,4,0,"Liverpool - Brighton & Hove Albion, 4 - 0",England
4,2500093,1611,1644,1611,1,0,"Manchester United - Watford, 1 - 0",England


In [3]:
arx = '/home/chacoma/Lineas/Futbol/Ranking/ranking-futbol/data/league2team2rank.json'
l2t2r = json.load( open(arx,"r") )

In [48]:
data= defaultdict(lambda : np.zeros((20,20)))

for row in df.itertuples():

    t1 = l2t2r[row.liga][str(row.home)]
    t2 = l2t2r[row.liga][str(row.away)]

    if (row.score_home>row.score_away):
        data[row.liga][ t1-1, t2-1] += 3

    elif (row.score_home==row.score_away):
        data[row.liga][ t1-1, t2-1] += 1
        data[row.liga][ t2-1, t1-1] += 1

    else:
        data[row.liga][ t2-1, t1-1] += 3


data["Germany"] = data["Germany"][:-2, :-2 ]


### ratings

In [53]:
def get_ratings_BradleyTerry( M ):

    Nt = M.shape[0]

    # 1. Inicialización
    beta = np.ones(Nt) / Nt
    Wi = np.sum(M, axis=1)  # Victorias totales de cada equipo (vector)

    tol = 1e-6
    error = 1.0
    max_iter = 1000
    iter_count = 0

    # 2. Loop de convergencia GLOBAL
    while error > tol and iter_count < max_iter:
        beta_viejo = beta.copy()
        beta_nuevo = np.zeros(Nt)
        
        for i in range(Nt):
            # Calculamos el denominador S para el equipo i
            # Usamos los betas de la iteración anterior
            S = 0
            for j in range(Nt):
                if i != j:
                    # El denominador suma 1 / (beta_i + beta_j)
                    S += 1.0 / (beta_viejo[i] + beta_viejo[j])
            
            # Evitamos división por cero si un equipo no tiene victorias
            if S > 0:
                beta_nuevo[i] = Wi[i] / S
            else:
                beta_nuevo[i] = 0.0
                
        # 3. Normalización (Crucial para estabilidad)
        beta_nuevo = beta_nuevo / np.sum(beta_nuevo)
        
        # 4. Cálculo del error (Norma infinito o Euclídea)
        error = np.linalg.norm(beta_nuevo - beta_viejo)
        beta = beta_nuevo
        iter_count += 1

    #print(f"Convergió en {iter_count} iteraciones")


    return np.log(beta)


# *************************************

S={}

for liga in ligas:

    print(liga)
    S[liga]={}


    M = data[liga]+0.01

    s= get_ratings_BradleyTerry( M )

    S[liga]=s


England
France
Germany
Italy
Spain


### rankings

In [55]:
R={}

for liga in ligas:
    print(liga)
    R[liga]={}

    R[liga]= np.argsort( -S[liga] )+1


England
France
Germany
Italy
Spain


In [61]:
R["France"]

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 16, 17, 15,
       18, 19, 20])